# 1 · Train end to end

One arm of the 2×2 study, from label file to checkpoint. Everything here is the
package doing the work — the notebook exists to make each step visible, not to
hold the logic.

Pick the arm by changing `CONFIG`. The four are `frozen_no_aug`, `frozen_aug`,
`unfrozen_no_aug`, `unfrozen_aug`; seeds 42/43/44.

For the command-line equivalent: `python scripts/train.py --config unfrozen_aug --seed 42`

In [ ]:
# --- environment ------------------------------------------------------------
# Local: nothing to do beyond `pip install -e .` in the repo root.
# Colab: uncomment, then set the two roots below.
#
# from google.colab import drive; drive.mount("/content/drive")
# !pip install -q -e "/content/drive/MyDrive/<path to repo>"

import os

# Where the SPEED+ download and the prepared subsets live.
os.environ.setdefault("SPEEDPLUS_DATA_ROOT", "/path/to/speedplusv2")
# Where checkpoints, predictions and figures go. On Colab put this on the
# mounted Drive so a disconnect does not cost the run.
os.environ.setdefault("SPEEDPLUS_ARTIFACTS", "/path/to/artifacts")

# On Colab, unpack image archives to local disk rather than reading tens of
# thousands of small files across the Drive mount. Set to None when local.
LOCAL_IMAGES = None  # e.g. "/content/images"

In [ ]:
CONFIG = "unfrozen_aug"
SEED = 42

from speedplus_pose.config import load_config, set_seed
from speedplus_pose.experiment import describe
from speedplus_pose.paths import Paths

config = load_config(CONFIG, SEED)
paths = Paths()

# Seed before anything builds a model, so initialisation is covered too.
set_seed(config.seed)

print(describe(config))
print(config.train)

## Labels

`load_labels` does one thing worth watching: it canonicalises every quaternion
into the `q1 ≥ 0` hemisphere. A rotation is represented by both **q** and **−q**,
so without this the network gets contradictory targets for identical rotations.

In [ ]:
from speedplus_pose.experiment import load_split_labels

train_labels = load_split_labels("train", paths)
validation_labels = load_split_labels("synthetic", paths)

print(f"{len(train_labels)} training, {len(validation_labels)} validation")
print("all quaternions canonicalised:", bool((train_labels["q1"] >= 0).all()))
train_labels.head()

## Translation normalisation

Translation targets are z-scored per component before the MSE term. The constants
come from the **training split only** — computing them over train+validation
would leak the validation distribution into the model — and they are saved into
every checkpoint so evaluation cannot silently use different ones.

In [ ]:
import torch

from speedplus_pose.labels import translation_normalisation
from speedplus_pose.model import resolve_device
from speedplus_pose.normalisation import TranslationNormaliser

device = resolve_device()
means, stds = translation_normalisation(train_labels)
normaliser = TranslationNormaliser(means, stds, device)

print(f"device: {device}")
print(f"means: {[round(m, 3) for m in means]}")
print(f"stds:  {[round(s, 3) for s in stds]}")

## Transforms

The training transform is the arm's second factor. With augmentation off it is
*identical* to the eval transform — the control arms train on exactly what they
are measured on.

With it on, the order matters: photometric jitter on the PIL image, then tensor
conversion, then the tensor-space augmentations, then normalise, then erasing.

In [ ]:
from speedplus_pose.transforms import build_eval_transform, build_train_transform

train_transform = build_train_transform(config.augmentation, config.image)
eval_transform = build_eval_transform(config.image)

for step in train_transform.transforms:
    print(" ", type(step).__name__)

In [ ]:
# Look at what the model will actually see. `preview=True` drops only the
# normalisation step, so the tensor still renders.
import matplotlib.pyplot as plt

from speedplus_pose.experiment import build_dataset

preview = build_dataset(
    "train", paths,
    build_train_transform(config.augmentation, config.image, preview=True),
    labels=train_labels.head(8), local_images=LOCAL_IMAGES,
)

figure, axes = plt.subplots(2, 4, figsize=(13, 4))
for ax, i in zip(axes.flat, range(8)):
    ax.imshow(preview[i][0].numpy()[0], cmap="gray")
    ax.set_axis_off()
figure.suptitle("What the network sees" + (" (augmented)" if config.augmentation.enabled else ""))
plt.tight_layout()

## Model

ResNet-50 with `fc` replaced by identity, so the head reads the pooled 2048-d
features rather than the 1000-d ImageNet logits.

`apply_freeze_policy` sets which parameters carry gradients. The trainable count
below is the cheap check that the arm is what you think it is: head only for a
frozen arm, head + layer 4 otherwise.

In [ ]:
from speedplus_pose.model import build_model, count_parameters

model = build_model(config.model, device)
total, trainable = count_parameters(model)

print(f"total     {total:,}")
print(f"trainable {trainable:,}  ({trainable / total:.1%})")

## Train

Two things in the loop are worth knowing about.

**`set_train_mode` is the arm definition, not bookkeeping.** A frozen arm keeps
the whole backbone in eval mode so its BatchNorm layers go on using the ImageNet
running statistics. An unfrozen arm puts layer 4 in train mode, so that stage's
statistics start tracking the training distribution — and with augmentation on,
that means the *augmented* distribution. That is the mechanism behind the study's
second finding. Calling `model.train()` alone would silently erase the
comparison.

**The loss is split.** Translation MSE plus a sign-invariant rotation term
`1 − ⟨q̂,q⟩²`. A single MSE over all seven outputs lets translation dominate and
rotation barely learns.

In [ ]:
from speedplus_pose.data import build_dataloader
from speedplus_pose.train import train

train_dataset = build_dataset("train", paths, train_transform, train_labels, LOCAL_IMAGES)
validation_dataset = build_dataset("synthetic", paths, eval_transform, validation_labels, LOCAL_IMAGES)

history = train(
    model=model,
    config=config,
    train_dataloader=build_dataloader(train_dataset, config.train, shuffle=True),
    validation_dataloader=build_dataloader(validation_dataset, config.train, shuffle=False),
    normaliser=normaliser,
    checkpoint_dir=paths.checkpoints(config.run_name),
    device=device,
)

In [ ]:
figure, axes = plt.subplots(1, 3, figsize=(14, 3.4))

axes[0].plot(history.train_losses)
axes[0].set_title("training loss")
axes[1].plot(history.validation_translation_errors)
axes[1].set_title("validation $E_T$")
axes[2].plot(history.validation_rotation_errors)
axes[2].set_title("validation $E_R$ [rad]")
for ax in axes:
    ax.set_xlabel("epoch")
    ax.grid(alpha=0.3)
plt.tight_layout()

## Select the checkpoint

**The selection rule, applied uniformly to every arm: best synthetic validation
rotation error.** No real-domain number may inform it. Record the chosen file in
`results/checkpoints.json` so the row it produces stays traceable.

In [ ]:
best = history.best_epoch()
print(f"best epoch {best + 1}: val_r {history.validation_rotation_errors[best]:.4f} rad, "
      f"val_t {history.validation_translation_errors[best]:.4f}")
print(f"\ncheckpoints in {paths.checkpoints(config.run_name)}:")
for path in sorted(paths.checkpoints(config.run_name).glob("checkpoint_*.pt")):
    print(" ", path.name)

Next: **02 · Measure the gap** takes a selected checkpoint and finds out what
survives contact with real imagery.